# Chapter 02 — Micrograd Value Class (Simplified)

Every modern neural network learns through backpropagation — the algorithm that computes how much each weight contributed to the model's error. But most practitioners treat it as a black box. In this chapter, you will build an automatic differentiation engine from scratch, giving you deep intuition about how neural networks actually learn.

### Glossary / Glossário

• backpropagation — algorithm that computes gradients by propagating errors backward through the network. / Algoritmo que computa gradientes propagando erros de trás para frente pela rede.
• autograd — automatic differentiation engine that tracks operations and computes gradients. / Mecanismo de diferenciação automática que rastreia operações e computa gradientes.
• gradient — derivative of the loss with respect to a parameter, indicating direction and magnitude of update. / Derivada do loss em relação a um parâmetro, indicando direção e magnitude da atualização.
• chain rule — calculus rule for computing derivatives of composed functions: dy/dx = dy/dg · dg/dx. / Regra do cálculo para derivadas de funções compostas: dy/dx = dy/dg · dg/dx.
• learning rate — scalar controlling how large each parameter update step is. / Escalar que controla o tamanho de cada passo de atualização de parâmetros.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

# === Neural network analogy ===
# In a real network: a = weight, b = input, c = output (part of loss)
# Gradients tell us: how much does each input affect the output?

a = Value(2.0)   # Think of this as a weight
b = Value(3.0)   # Think of this as an input
c = a * b        # Output: weight × input = 6.0
c.backward()

print("=== Values (forward pass) ===")
print(f"  a (weight) = {a.data}")
print(f"  b (input)  = {b.data}")
print(f"  c (output) = a × b = {c.data}")
print()
print("=== Gradients (backward pass) ===")
print(f"  dc/da = {a.grad}  <- how much 'a' affects 'c' (equals b)")
print(f"  dc/db = {b.grad}  <- how much 'b' affects 'c' (equals a)")

# Random inputs loop: gradients change every step
import random
random.seed(42)
print(f"\n{'Step':>4s} | {'a(wt)':>6s} | {'b(in)':>6s} | {'c=a*b':>7s} | {'dc/da':>6s} | {'dc/db':>6s}")
print("-" * 52)
for step in range(5):
    a = Value(round(random.uniform(1.0, 5.0), 2))
    b = Value(round(random.uniform(1.0, 5.0), 2))
    c = a * b
    c.backward()
    print(f"{step:>4d} | {a.data:>6.2f} | {b.data:>6.2f} | {c.data:>7.2f} | {a.grad:>6.2f} | {b.grad:>6.2f}")
print("\nNotice: dc/da always equals b, dc/db always equals a — but values change each step!")

# === Gradient descent: the weight learns! ===
# Goal: adjust weight 'a' so that c = a * b reaches target = 10.0
lr = 0.01
target = 10.0

a = Value(2.0)   # weight (learnable) — starts at 2.0, should converge to 3.33
b = Value(3.0)   # input (fixed)

print(f"\n{'Step':>4s} | {'a (weight)':>10s} | {'c = a*b':>8s} | {'loss':>8s} | {'dc/da':>6s}")
print("-" * 50)
for step in range(8):
    c = a * b
    loss = (c.data - target) ** 2
    c.backward()
    print(f"{step:>4d} | {a.data:>10.4f} | {c.data:>8.4f} | {loss:>8.4f} | {a.grad:>6.2f}")
    a.data -= lr * a.grad   # gradient descent update
    a.grad = 0.0            # zero the gradient for next step

print(f"\nWeight 'a' converged from 2.0 toward {target/b.data:.2f} (= target/b)")
print(f"This is how neural networks learn: forward -> backward -> update, repeat!")

---

**Course:** Aprenda LLM | **Chapter 02**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.